In [2]:
!pip install faiss-cpu -q

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv('datasets/train.csv')

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")


Creating knowledge base


Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created


In [3]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)
row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

result = zs(prompt_150, candidate_labels=labels_150)

print("Predicted labels (sorted by score):")
for label, score in zip(result['labels'], result['scores']):
    print(f"{score:.3f}  -->  {label[:80]}")

print("\nGround truth correct option:", row_150['answer'])
print("Ground truth text:", ans_150[:100])

# Find the score assigned to the correct answer
correct_idx = result['labels'].index(ans_150)
correct_score = result['scores'][correct_idx]
print(f"\nQ1 Answer - Probability score for ground-truth correct option: {correct_score:.3f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Predicted labels (sorted by score):
0.384  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.379  -->  The butterfly effect is the phenomenon that a large change in the initial condit
0.093  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.079  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.066  -->  The butterfly effect is the phenomenon that a large change in the initial condit

Ground truth correct option: C
Ground truth text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical 

Q1 Answer - Probability score for ground-truth correct option: 0.384


In [4]:
prompt_emb_150 = model.encode([prompt_150])

k = 10
distances, retrieved_indices = index.search(prompt_emb_150, k)

print("Retrieved indices (top 10):", retrieved_indices[0])
print("Distances:", distances[0])

# Find at what rank (1-indexed) the true document (index 150) appears
retrieved_list = retrieved_indices[0].tolist()

if 150 in retrieved_list:
    rank = retrieved_list.index(150) + 1  # 1-indexed
    print(f"\nQ2 Answer - True document found at rank: {rank}")
else:
    print("\nQ2 Answer - True document (index 150) NOT found in top 10")

Retrieved indices (top 10): [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Distances: [0.26394257 0.26394257 0.2664284  0.2664284  0.2686218  0.26993614
 0.26993614 0.26993614 0.28316098 0.28714597]

Q2 Answer - True document found at rank: 10


In [5]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices[0]]  # top 10 chunks from Q2
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort indices by cross-encoder score, descending
sorted_order = np.argsort(ce_scores)[::-1]
sorted_original_indices = [retrieved_indices[0][i] for i in sorted_order]

print("Cross-encoder reranked order (original KB indices):", sorted_original_indices)
print("Cross-encoder scores (sorted):", [ce_scores[i] for i in sorted_order])

if 150 in sorted_original_indices:
    ce_rank = sorted_original_indices.index(150) + 1
    print(f"\nQ3 Answer - True document reranked to position: {ce_rank}")
else:
    print("\nQ3 Answer - True document not found")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder reranked order (original KB indices): [np.int64(150), np.int64(1906), np.int64(847), np.int64(1693), np.int64(1269), np.int64(1532), np.int64(168), np.int64(576), np.int64(1701), np.int64(663)]
Cross-encoder scores (sorted): [np.float32(4.758512), np.float32(4.752577), np.float32(4.752577), np.float32(4.752577), np.float32(4.737523), np.float32(4.737523), np.float32(4.7072477), np.float32(4.687043), np.float32(4.6602316), np.float32(4.6602316)]

Q3 Answer - True document reranked to position: 1


In [6]:
prompt_42 = str(train.iloc[42]['prompt'])
prompt_42_emb = model.encode([prompt_42])

k5 = 5
distances_42, retrieved_indices_42 = index.search(prompt_42_emb, k5)

docs_5 = [kb[i] for i in retrieved_indices_42[0]]
concatenated_docs = " ".join(docs_5)

rag_string_42 = f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens = tokenizer(rag_string_42, truncation=False)
num_tokens = len(tokens['input_ids'])

print("Concatenated docs preview:", concatenated_docs[:200])
print("\nRAG string length (chars):", len(rag_string_42))
print(f"\nQ4 Answer - Total tokens: {num_tokens}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Concatenated docs preview: Permutation-inversion groups are groups of symmetry operations that are energetically feasible permutations of identical nuclei or inversion with respect to the center of mass, or a combination of bot

RAG string length (chars): 1127

Q4 Answer - Total tokens: 216


In [7]:
true_doc_150 = kb[150]

rag_string_150 = f"Context: {true_doc_150} Question: {prompt_150}"

result_rag = zs(rag_string_150, candidate_labels=labels_150)

print("Predicted labels (sorted by score) WITH RAG context:")
for label, score in zip(result_rag['labels'], result_rag['scores']):
    print(f"{score:.3f}  -->  {label[:80]}")

correct_idx_rag = result_rag['labels'].index(ans_150)
correct_score_rag = result_rag['scores'][correct_idx_rag]

print(f"\nQ5 Answer - New probability score for ground-truth correct option: {correct_score_rag:.3f}")
print(f"(Compare to Q1 baseline without RAG: 0.384)")

Predicted labels (sorted by score) WITH RAG context:
0.989  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.004  -->  The butterfly effect is the phenomenon that a large change in the initial condit
0.003  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.002  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.002  -->  The butterfly effect is the phenomenon that a large change in the initial condit

Q5 Answer - New probability score for ground-truth correct option: 0.989
(Compare to Q1 baseline without RAG: 0.384)


In [8]:
adversarial_doc = kb[999]

rag_string_adversarial = f"Context: {adversarial_doc} Question: {prompt_150}"

result_adv = zs(rag_string_adversarial, candidate_labels=labels_150)

print("Adversarial document injected:", adversarial_doc[:150])
print("\nPredicted labels (sorted by score) WITH ADVERSARIAL context:")
for label, score in zip(result_adv['labels'], result_adv['scores']):
    print(f"{score:.3f}  -->  {label[:80]}")

correct_idx_adv = result_adv['labels'].index(ans_150)
correct_score_adv = result_adv['scores'][correct_idx_adv]

print(f"\nQ6 Answer - Probability of correct option with adversarial context: {correct_score_adv:.3f}")
print(f"(Compare: no RAG = 0.384, good RAG = 0.989, bad RAG = {correct_score_adv:.3f})")

Adversarial document injected: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal te

Predicted labels (sorted by score) WITH ADVERSARIAL context:
0.529  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.425  -->  The butterfly effect is the phenomenon that a large change in the initial condit
0.020  -->  The butterfly effect is the phenomenon that a small change in the initial condit
0.019  -->  The butterfly effect is the phenomenon that a large change in the initial condit
0.007  -->  The butterfly effect is the phenomenon that a small change in the initial condit

Q6 Answer - Probability of correct option with adversarial context: 0.529
(Compare: no RAG = 0.384, good RAG = 0.989, bad RAG = 0.529)


In [9]:
hits = 0

for idx in range(100):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_text = str(row[row['answer']])

    prompt_emb = model.encode([prompt])
    _, retrieved_idx = index.search(prompt_emb, 5)

    retrieved_docs = [kb[i] for i in retrieved_idx[0]]

    if correct_text in retrieved_docs:
        hits += 1

hit_rate = (hits / 100) * 100
print(f"Q7 Answer - Hit Rate: {hit_rate:.1f}%")

Q7 Answer - Hit Rate: 73.0%


In [12]:
def apk(actual, predicted, k=3):
    if len(predicted) > k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i, p in enumerate(predicted):
        if p == actual:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
            break
    return score

In [13]:
map3_scores = []

for idx in range(20):
    row = train.iloc[idx]
    prompt = str(row['prompt'])
    correct_answer = row['answer']
    option_cols = ['A', 'B', 'C', 'D', 'E']
    labels = [str(row[c]) for c in option_cols]

    # Retrieve top 5
    prompt_emb = model.encode([prompt])
    _, retrieved_idx = index.search(prompt_emb, 5)
    docs_5 = [kb[i] for i in retrieved_idx[0]]

    # Rerank with cross-encoder, pick best doc
    pairs = [[prompt, doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = docs_5[np.argmax(ce_scores)]

    # Augment
    rag_string = f"Context: {best_doc} Question: {prompt}"

    # Predict
    result = zs(rag_string, candidate_labels=labels)

    # Map predicted label text back to letter, rank descending
    label_to_letter = {str(row[c]): c for c in option_cols}
    ranked_letters = [label_to_letter[lbl] for lbl in result['labels']]
    top3 = ranked_letters[:3]

    # Score this row's MAP@3
    score = apk(correct_answer, top3, k=3)
    map3_scores.append(score)

    print(f"Row {idx}: correct={correct_answer}, top3={top3}, score={score:.3f}")

final_map3 = np.mean(map3_scores)
print(f"\nQ8 Answer - Final average MAP@3 across 20 rows: {final_map3:.3f}")

Row 0: correct=B, top3=['B', 'D', 'A'], score=1.000
Row 1: correct=A, top3=['A', 'E', 'C'], score=1.000
Row 2: correct=C, top3=['C', 'D', 'B'], score=1.000


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Row 3: correct=B, top3=['B', 'D', 'A'], score=1.000
Row 4: correct=A, top3=['A', 'B', 'C'], score=1.000
Row 5: correct=C, top3=['B', 'C', 'A'], score=0.500
Row 6: correct=E, top3=['E', 'B', 'D'], score=1.000
Row 7: correct=A, top3=['A', 'B', 'C'], score=1.000
Row 8: correct=A, top3=['A', 'C', 'D'], score=1.000
Row 9: correct=A, top3=['A', 'B', 'C'], score=1.000
Row 10: correct=C, top3=['C', 'A', 'D'], score=1.000
Row 11: correct=B, top3=['B', 'C', 'A'], score=1.000
Row 12: correct=D, top3=['D', 'A', 'B'], score=1.000
Row 13: correct=E, top3=['E', 'D', 'B'], score=1.000
Row 14: correct=E, top3=['E', 'A', 'D'], score=1.000
Row 15: correct=E, top3=['E', 'B', 'C'], score=1.000
Row 16: correct=C, top3=['C', 'B', 'E'], score=1.000
Row 17: correct=C, top3=['C', 'B', 'E'], score=1.000
Row 18: correct=B, top3=['B', 'D', 'A'], score=1.000
Row 19: correct=C, top3=['C', 'D', 'A'], score=1.000

Q8 Answer - Final average MAP@3 across 20 rows: 0.975
